# ColabのL4 GPUでvLLMを立てる

ColabのL4 GPU上のvLLMで `openai/gpt-oss-20b` を起動し、手元から OpenAI 互換API（`/v1/chat/completions` など）を叩けるようにする。RAGをOllamaからvLLMへ移すための動作・速度確認用で、RAGアプリ（`rag_web` と `ingest/`）はまだOllamaのAPI（`/api/chat` など）を使うため、このノートブックのURLを `.env` に入れてもRAGは動かない。

**手順**
1. ランタイム → ランタイムのタイプを変更 → GPU（L4）を選択してから、上から順にセルを実行する
2. ngrokの無料アカウントを作り、ダッシュボードでauthtokenを控えておく（未取得なら ngrok.com で登録できる）
3. セル7で表示されるURLとセル4のAPIキーを使い、末尾の「手元から試す」の手順で呼び出す

**構成**: ngrok → 認証プロキシ（8000番、`X-API-Key` を検証）→ vLLM（8001番）。Ollama版（`run_ollama_server.ipynb`）と同じ認証の流儀（`X-API-Key` ヘッダー）にしてあり、RAG側をvLLMへ切り替えたあとは `.env` の `OLLAMA_HOST` / `OLLAMA_API_KEY` の形で使える見込みである。**切り替えるまでは、このURLとキーを `.env` に入れないこと**（RAGの生成・取り込みが失敗する）。

**注意**
- Colabのランタイムはアイドルや時間経過で切断される。常時稼働のAPIではなく、使う時だけ起動するものとして扱うこと。
- ngrokのURLは誰でも到達できるが、`X-API-Key` ヘッダーが一致しないリクエストはプロキシが401で拒否する。それでもURLとAPIキーは他人に共有しないこと。
- 初回はvLLMの導入（数分）とモデルのダウンロード（約13GB）で時間がかかる。

In [ ]:
# 0. 設定。後続のセルはここの値を使う（値を変えたらこのセルから実行し直す）。
import os

VLLM_VENV = "/content/vllm-env"
VLLM_MODEL = "openai/gpt-oss-20b"
# プロキシが8000番を使うため、vLLMは8001番で待ち受ける
VLLM_PORT = 8001
PROXY_PORT = 8000
# L4のVRAM 24GBを前提にした値。起動時に CUDA out of memory で落ちる場合は
# VLLM_MAX_MODEL_LEN を 32768 に下げる。
VLLM_MAX_MODEL_LEN = 65536
VLLM_GPU_MEMORY_UTILIZATION = "0.90"
# 初回はモデル（約13GB）のダウンロードとCUDAグラフの準備で10分以上かかることがある
VLLM_START_TIMEOUT_SECONDS = 1800


def vllm_env() -> dict:
    """vLLMの仮想環境を有効化したのと同じ環境変数。

    vllm download-kernels は「同じPython環境で」実行する前提で、vllm serve も
    JITコンパイル用のツール（ninja等）を仮想環境の bin から探す。
    """
    return {
        **os.environ,
        "VIRTUAL_ENV": VLLM_VENV,
        "PATH": f"{VLLM_VENV}/bin:" + os.environ.get("PATH", ""),
    }

In [ ]:
# 1. GPU（L4）を確認する。
# セル0の値はL4のVRAM 24GBを前提に決めてある。T4（16GB）などが割り当てられると
# vLLMの起動途中で分かりにくい失敗をするため、ここで先に止める。
import subprocess

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
gpu_info = gpu_check.stdout.strip()
if gpu_check.returncode != 0 or not gpu_info:
    raise RuntimeError(
        "GPUが見えていません。ランタイムのタイプがGPU（L4）か確認してください: "
        + gpu_check.stderr
    )
if "L4" not in gpu_info:
    raise RuntimeError(
        f"L4以外のGPUが割り当てられています（{gpu_info}）。"
        "ランタイム → ランタイムのタイプを変更 でL4を選び直してください。"
    )
print("GPU確認OK:", gpu_info)

In [ ]:
# 2. vLLMを専用の仮想環境へ入れる（数分かかる）。
# Colab同梱の torch と vLLM が要求する torch のバージョンが食い違うため、
# Colab本体の環境には入れず VLLM_VENV に分ける。--allow-existing は再実行時に
# 既存の仮想環境を作り直さずに使い回すため。--seed は仮想環境に pip を入れるため
# （vLLM公式の uv 手順と同じ。download-kernels が追加パッケージを入れる際に要る）。
import subprocess

subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(
    ["uv", "venv", VLLM_VENV, "--python", "3.12", "--seed", "--allow-existing"], check=True
)
subprocess.run(
    ["uv", "pip", "install", "--python", f"{VLLM_VENV}/bin/python", "vllm", "--torch-backend=auto"],
    check=True,
)
# FlashInferの事前コンパイル済みカーネルを取得し、起動時のコンパイル待ちを避ける
subprocess.run([f"{VLLM_VENV}/bin/vllm", "download-kernels"], check=True, env=vllm_env())
print("vLLMの導入OK")

In [ ]:
# 3. vLLMをバックグラウンドで起動し、/v1/models が応答するまで待つ。
# 起動途中で落ちた場合（VRAM不足など）は待ち時間を使い切らず、すぐにログの
# 末尾を表示して止める。ログ全体は vllm.log にある。
import subprocess
import time
from pathlib import Path

import requests

VLLM_LOG_PATH = Path("vllm.log")

# 再実行時、前回のvLLMがGPUメモリと8001番を掴んだままだと新しいプロセスが起動に失敗する
if "vllm_proc" in globals() and vllm_proc.poll() is None:
    vllm_proc.terminate()
    vllm_proc.wait(timeout=60)

VLLM_COMMAND = [
    f"{VLLM_VENV}/bin/vllm",
    "serve",
    VLLM_MODEL,
    "--port",
    str(VLLM_PORT),
    "--max-model-len",
    str(VLLM_MAX_MODEL_LEN),
    "--gpu-memory-utilization",
    VLLM_GPU_MEMORY_UTILIZATION,
]
vllm_log = VLLM_LOG_PATH.open("w", encoding="utf-8")
vllm_proc = subprocess.Popen(
    VLLM_COMMAND, stdout=vllm_log, stderr=subprocess.STDOUT, env=vllm_env()
)


def show_vllm_log_tail():
    # V1エンジンでは本当の原因（CUDA out of memory 等）がエンジン側の子プロセスから
    # 先に出て、その後にAPIサーバー側の長いトレースバック（"See root cause above"）が
    # 続く。末尾だけでは原因が見えないため、エラーらしき行を先に抜き出して見せる。
    vllm_log.flush()
    text = VLLM_LOG_PATH.read_text(encoding="utf-8", errors="replace")
    suspects = [
        line
        for line in text.splitlines()
        if "Error" in line or "out of memory" in line.lower()
    ]
    print("--- vllm.log のエラー行 ---")
    print("\n".join(suspects[-30:]) or "（該当なし）")
    print("--- vllm.log（末尾） ---")
    print(text[-3000:])


for _ in range(VLLM_START_TIMEOUT_SECONDS // 5):
    if vllm_proc.poll() is not None:
        show_vllm_log_tail()
        raise RuntimeError(
            f"vLLMが起動途中で終了しました（終了コード {vllm_proc.returncode}）。上のログを確認してください"
        )
    try:
        if requests.get(f"http://127.0.0.1:{VLLM_PORT}/v1/models", timeout=5).status_code == 200:
            print("vLLM起動確認OK")
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    show_vllm_log_tail()
    raise RuntimeError(f"vLLMが{VLLM_START_TIMEOUT_SECONDS}秒以内に応答しませんでした")

In [ ]:
# 4. プロキシ用のAPIキーを発行する。手元から呼ぶときに X-API-Key ヘッダーへ入れる。
# RAGアプリはまだOllamaのAPIを使うため、このキーとURLを手元の .env に入れてはいけない
# （入れるとRAGの生成・取り込みが失敗し、取り込み時は資料本文がColabへ送られてから落ちる）。
import os
import secrets

PROXY_API_KEY = secrets.token_urlsafe(24)
# プロキシは起動時にこの環境変数を読む。生成ファイルへキーを埋め込まない。
os.environ["PROXY_API_KEY"] = PROXY_API_KEY
print("APIキー（手元から試す際に X-API-Key ヘッダーへ入れる）:")
print(PROXY_API_KEY)

In [ ]:
# 5. リバースプロキシ本体を書き出す。
# X-API-Keyヘッダーを検証したうえで、パスをそのままvLLMへストリーミング転送する
# （/v1/chat/completions のSSEストリームもそのまま通す）。
# run_ollama_server.ipynb のプロキシの写しで、違いはキーの環境変数名・転送先・
# エラー文言だけ。Colabからリポジトリを読まない構成のため写しにしている
# （技術的負債: プロキシを直すときは両方のノートブックを直すこと）。
from pathlib import Path

proxy_code = """
import os

import httpx
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse

API_KEY = os.environ["PROXY_API_KEY"]
VLLM_BASE = os.environ.get("VLLM_UPSTREAM", "http://127.0.0.1:__VLLM_PORT__")

app = FastAPI()
_client = httpx.AsyncClient(
    base_url=VLLM_BASE,
    timeout=httpx.Timeout(connect=10.0, read=600.0, write=60.0, pool=10.0),
)

_HOP_BY_HOP = {"host", "x-api-key", "authorization", "content-length", "connection", "transfer-encoding"}


@app.api_route("/{path:path}", methods=["GET", "POST"])
async def proxy(path: str, request: Request):
    if request.headers.get("x-api-key") != API_KEY:
        return JSONResponse({"error": "unauthorized"}, status_code=401)

    body = await request.body()
    upstream_req = _client.build_request(
        request.method,
        f"/{path}",
        params=request.query_params,
        content=body,
        headers=[(k, v) for k, v in request.headers.items() if k.lower() not in _HOP_BY_HOP],
    )
    try:
        upstream = await _client.send(upstream_req, stream=True)
    except httpx.TimeoutException:
        return JSONResponse({"error": "vLLM upstream timed out"}, status_code=504)
    except httpx.RequestError:
        return JSONResponse({"error": "Unable to reach vLLM upstream"}, status_code=502)

    async def body_iter():
        try:
            async for chunk in upstream.aiter_raw():
                yield chunk
        finally:
            await upstream.aclose()

    return StreamingResponse(
        body_iter(),
        status_code=upstream.status_code,
        headers={k: v for k, v in upstream.headers.items() if k.lower() not in _HOP_BY_HOP},
    )
""".replace("__VLLM_PORT__", str(VLLM_PORT))

Path("vllm_proxy.py").write_text(proxy_code, encoding="utf-8")
print("vllm_proxy.py を書き出しました")

In [ ]:
# 6. 認証プロキシに必要なパッケージを入れ、プロキシをバックグラウンドで起動する。
# プロキシはColab本体の環境で動かす（vLLMの仮想環境とは別。vLLMとはHTTPでしか話さない）。
# セル4(APIキー再発行)やこのセル自体を再実行すると、直前に起動したプロキシが
# 8000番を掴んだまま古いAPIキーで生き続け、新しいプロキシがバインドに失敗して
# 即終了する。再実行しても安全なように、起動前に前回のプロセスを必ず止める。
import subprocess
import time
from pathlib import Path

import requests

subprocess.run(["pip", "install", "-q", "fastapi", "uvicorn[standard]", "httpx", "pyngrok"], check=True)

if "proxy_proc" in globals() and proxy_proc.poll() is None:
    proxy_proc.terminate()
    proxy_proc.wait(timeout=10)

proxy_log_path = Path("proxy.log")
proxy_log = proxy_log_path.open("w", encoding="utf-8")
proxy_proc = subprocess.Popen(
    ["uvicorn", "vllm_proxy:app", "--host", "0.0.0.0", "--port", str(PROXY_PORT)],
    stdout=proxy_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        r = requests.get(
            f"http://127.0.0.1:{PROXY_PORT}/v1/models",
            headers={"X-API-Key": PROXY_API_KEY},
            timeout=2,
        )
        if r.status_code == 200:
            print("プロキシ起動確認OK（vLLMまで疎通）")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    proxy_log.flush()
    print("--- proxy.log ---")
    print(proxy_log_path.read_text(encoding="utf-8", errors="replace"))
    raise RuntimeError("プロキシの起動に失敗しました")

In [ ]:
# 7. ngrokでプロキシ（8000番）を外部公開する。
# authtokenはngrokダッシュボード（https://dashboard.ngrok.com/get-started/your-authtoken）から取得する
from getpass import getpass

from google.colab import userdata
from pyngrok import conf, ngrok

# Colabのシークレット（鍵アイコン）に NGROK_AUTHTOKEN があれば使い、毎回の入力を省く。
# 未登録・アクセス未許可のときだけ入力に切り替える。それ以外の失敗は隠さない。
try:
    ngrok_token = (userdata.get("NGROK_AUTHTOKEN") or "").strip()
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    ngrok_token = ""

if not ngrok_token:
    ngrok_token = getpass("ngrok authtoken: ").strip()

if not ngrok_token:
    raise ValueError("ngrok authtoken が入力されていません。")

conf.get_default().auth_token = ngrok_token

# 既存のトンネルがあれば閉じてから張り直す（再実行時の重複防止）
ngrok.kill()
public_url = ngrok.connect(PROXY_PORT, "http").public_url

print("URL（手元から試す際に使う。RAGアプリ用の設定ファイルには入れない）:")
print(public_url)

In [ ]:
# 8. 動作と速度を確かめる。プロキシ経由（ngrokの手前まで同じ経路）でストリーミング生成し、
# 初トークンまでの時間・全体の時間・トークン/秒と、思考（reasoning）と回答（content）が
# それぞれ届いたかを表示する。Ollama版と比べるときは同じ質問を両方に投げる。
import json
import time

import requests


def measure_chat(base_url: str, api_key: str, prompt: str, max_tokens: int = 1024) -> dict:
    started = time.perf_counter()
    response = requests.post(
        f"{base_url}/v1/chat/completions",
        headers={"X-API-Key": api_key},
        json={
            "model": VLLM_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": True,
            "stream_options": {"include_usage": True},
            "max_tokens": max_tokens,
        },
        stream=True,
        timeout=600,
    )
    response.raise_for_status()

    first_token_at = None
    reasoning_chars = 0
    content_parts = []
    completion_tokens = None
    for line in response.iter_lines(decode_unicode=True):
        if not line or not line.startswith("data: "):
            continue
        data = line[len("data: "):]
        if data == "[DONE]":
            break
        chunk = json.loads(data)
        if chunk.get("usage"):
            completion_tokens = chunk["usage"].get("completion_tokens")
        for choice in chunk.get("choices", []):
            delta = choice.get("delta", {})
            # vLLMのバージョンによって思考のフィールド名が reasoning / reasoning_content と異なる
            reasoning = delta.get("reasoning") or delta.get("reasoning_content") or ""
            content = delta.get("content") or ""
            if (reasoning or content) and first_token_at is None:
                first_token_at = time.perf_counter()
            reasoning_chars += len(reasoning)
            content_parts.append(content)

    total_seconds = time.perf_counter() - started
    return {
        "first_token_seconds": None if first_token_at is None else first_token_at - started,
        "total_seconds": total_seconds,
        "completion_tokens": completion_tokens,
        # 初トークンまでの待ち（プロンプト処理）を除いた生成速度。Ollamaの eval rate と同じ定義
        "decode_tokens_per_second": (
            None
            if completion_tokens is None or first_token_at is None
            else completion_tokens / (total_seconds - (first_token_at - started))
        ),
        "reasoning_chars": reasoning_chars,
        "content": "".join(content_parts),
    }


speed_result = measure_chat(
    f"http://127.0.0.1:{PROXY_PORT}",
    PROXY_API_KEY,
    "日本の首都はどこですか？一文で答えてください。",
)
for key, value in speed_result.items():
    print(f"{key}: {value}")
if not speed_result["content"]:
    print(
        "回答（content）が空です。思考（reasoning）だけで max_tokens を使い切った可能性があります。"
        "max_tokens を増やして再実行してください。"
    )

## 手元から試す

セル7のURLとセル4のAPIキーを使い、手元の PowerShell から呼び出す。`X-API-Key` ヘッダーが無いと401になる。

```powershell
$url = "https://xxxx.ngrok-free.app"   # セル7で表示されたURL
$key = "セル4で表示されたAPIキー"
$headers = @{ "X-API-Key" = $key }

# モデル一覧
Invoke-RestMethod -Uri "$url/v1/models" -Headers $headers

# 生成（非ストリーム）。gpt-oss は思考を message.reasoning（版により reasoning_content）、回答を message.content に分けて返す
$body = @{
    model = "openai/gpt-oss-20b"
    messages = @(@{ role = "user"; content = "日本の首都はどこですか？" })
    max_tokens = 1024
} | ConvertTo-Json -Depth 5
$res = Invoke-RestMethod -Uri "$url/v1/chat/completions" -Method Post -Headers $headers `
    -ContentType "application/json; charset=utf-8" -Body ([Text.Encoding]::UTF8.GetBytes($body))
$res.choices[0].message.content
```

**RAGアプリについて**: RAGアプリ（`rag_web` と `ingest/`）はまだOllamaのネイティブAPI（`/api/chat`・`/api/embed`・`/api/tags`）を使うため、このURLを `.env` の `OLLAMA_HOST` に入れてもRAGは動かない。埋め込み（bge-m3）とVLM（qwen3-vl）もこのノートブックでは立てていない。

## 後片付け

GPU使用量とログを確認し、使い終わったらランタイムを切断する（ランタイム → ランタイムを接続解除して削除）。

```python
!nvidia-smi
print(Path("vllm.log").read_text(encoding="utf-8", errors="replace")[-3000:])
```